# Experiment 5.4.3 — Causal elapsed-time readout capacity

**Analysis-only notebook.** Training, selection, and final-test execution live in the Python/Slurm pipeline. This notebook only reads finalized CSV/JSON artifacts and never regenerates missing runs.

Question: can a compact causal elapsed-time-conditioned readout recover the phase-specific weighting of frozen-WHAT Fixed250 + full Linear?


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('notebooks/artifacts/experiment_5_4_3_elapsed_readout_capacity/elapsed_readout_capacity_v1')
if not ROOT.exists():
    ROOT = Path('artifacts/experiment_5_4_3_elapsed_readout_capacity/elapsed_readout_capacity_v1')
ROOT


In [ ]:
required = ['reference_runs.csv', 'capacity_runs.csv', 'capacity_summary.csv', 'capacity_recovery.csv', 'capacity_selection.json']
missing = [name for name in required if not (ROOT / name).exists()]
if missing:
    raise FileNotFoundError(f'Missing finalized Stage-A artifacts: {missing}')
reference = pd.read_csv(ROOT / 'reference_runs.csv')
capacity = pd.read_csv(ROOT / 'capacity_runs.csv')
capacity_summary = pd.read_csv(ROOT / 'capacity_summary.csv')
recovery = pd.read_csv(ROOT / 'capacity_recovery.csv')
capacity_selection = json.loads((ROOT / 'capacity_selection.json').read_text())

summary_parts = [capacity_summary.assign(source_stage='capacity')]
if (ROOT / 'extension_summary.csv').exists():
    extension_summary = pd.read_csv(ROOT / 'extension_summary.csv')
    summary_parts.append(extension_summary.assign(source_stage='extension'))
summary = pd.concat(summary_parts, ignore_index=True, sort=False)
selection = json.loads((ROOT / 'selection.json').read_text()) if (ROOT / 'selection.json').exists() else None

display(reference)
display(summary.sort_values('mean_val_balanced_accuracy', ascending=False))
print('Stage-A decision:')
display(capacity_selection)
if selection is not None:
    print('Locked validation-only selection:')
    display(selection)


In [ ]:
pivot = capacity_summary[capacity_summary.parameterization == 'factorized'].pivot(index='rank', columns='n_banks', values='mean_val_balanced_accuracy')
fig, ax = plt.subplots(figsize=(7, 4.5))
image = ax.imshow(pivot.values, aspect='auto')
ax.set_xticks(range(len(pivot.columns)), [str(v) for v in pivot.columns])
ax.set_yticks(range(len(pivot.index)), [str(v) for v in pivot.index])
ax.set_xlabel('Number of banks K')
ax.set_ylabel('Rank r')
ax.set_title('Stage-A validation BA — elapsed-conditioned capacity map')
fig.colorbar(image, ax=ax, label='Balanced accuracy')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ordered = summary.sort_values('trainable_parameter_count')
ax.scatter(ordered.trainable_parameter_count, ordered.mean_oracle_gap_recovery_fraction)
for _, row in ordered.iterrows():
    label = f"{row.parameterization}:K{int(row.n_banks)}/r{int(row['rank'])}"
    ax.annotate(label, (row.trainable_parameter_count, row.mean_oracle_gap_recovery_fraction))
ax.axhline(0.90, linestyle='--')
ax.set_xlabel('Trainable parameters')
ax.set_ylabel('Mean Fixed250 oracle-gap recovery fraction')
ax.set_title('Validation capacity efficiency')
plt.show()


In [ ]:
if (ROOT / 'final_runs.csv').exists():
    final = pd.read_csv(ROOT / 'final_runs.csv')
    display(final)
    cols = ['base_test_balanced_accuracy', 'fixed250_test_balanced_accuracy', 'anchor_k4_r4_test_balanced_accuracy', 'selected_test_balanced_accuracy', 'oracle_gap_recovery_fraction']
    display(final[cols].agg(['mean', 'std']))

    if (ROOT / 'gate_activity.csv').exists():
        gate = pd.read_csv(ROOT / 'gate_activity.csv')
        gate_time = gate[gate.metric == 'bank_probability_by_elapsed_time']
        if len(gate_time):
            gate_mean = gate_time.groupby(['elapsed_seconds', 'bank'], as_index=False).value.mean()
            fig, ax = plt.subplots(figsize=(8, 4.5))
            for bank, frame in gate_mean.groupby('bank'):
                ax.plot(frame.elapsed_seconds, frame.value, marker='o', label=f'bank {int(bank)}')
            ax.set_xlabel('Elapsed time (s)')
            ax.set_ylabel('Mean gate probability')
            ax.set_title('Selected readout gate schedule')
            ax.legend(ncol=2)
            plt.show()

    if (ROOT / 'effective_weight_distance.csv').exists():
        distance = pd.read_csv(ROOT / 'effective_weight_distance.csv')
        display(distance.groupby('bin_index')[['normalized_frobenius_distance', 'cosine_similarity']].mean())
else:
    print('Final test remains unopened / not yet finalized.')
